# Batch API synthesis: how many of six Arrhenius parameters can four batches support?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/03_estimability/tutorial.ipynb)

**Estimability analysis** (`estima`: orthogonal ranking and the rCC criterion).

Four batches already exist. Fitting all six parameters to one or two of them gives huge, correlated
intervals; fixing the right ones gives a smaller model that the data *can* support. `estima`

1. ranks the parameters from most to least estimable (orthogonalisation of the scaled
   sensitivity matrix), and
2. picks how many of them to estimate with the corrected critical ratio **rCC**, which balances the
   bias of fixing parameters against the variance of estimating them.

You will add the batches one at a time and watch the number of estimable parameters grow.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['model.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/03_estimability/' + f, f)

### The process

Batch synthesis of a quaternary ammonium salt (an intermediate of an HIV drug):

$$\mathrm{SM + TMA \rightleftharpoons QS1Cl \qquad QS1Cl \rightarrow ClDMI + MeCl}$$

The product QS1Cl forms reversibly and slowly degrades into the impurity ClDMI. Rate and
equilibrium constants follow Arrhenius / van 't Hoff laws around 296.15 K, so the model has six
parameters: three constants at the reference temperature and three energies. Measured: starting
material `y1`, product `y2`, impurity `y3`. Inputs: the temperature profile `u1` and the initial
concentrations `y10`, `y20`.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds
from middoe.sc_estima import estima
from middoe.log_utils import save_to_jac, load_from_jac
from middoe.iden_utils import run_postprocessing

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.
HERE = Path.cwd()
# Start clean: expera() ADDS sheets to an existing data.xlsx, so remove what an earlier run wrote.
for old in ['data.xlsx', 'data_plots', 'modelling', 'confidence', 'post_processing', 'estimability',
            'meshindep_plots', 'design_plots', 'iden_results.jac']:
    p = HERE / old
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
np.random.seed(0)                                # same noise at every run

In [ ]:
theta_true  = [50000, 75000, 0.4116, 111900, 9905, 30000]   # the in-silico 'real' process
theta_guess = [100000, 100000, 1, 100000, 100, 10000]       # starting values
theta_min   = [10000, 0, 0.1, 50000, 10, 10000]             # bounds
theta_max   = [1000000, 200000, 10, 200000, 10000, 200000]
NAMES = ['kf_ref', 'Ef', 'kfs_ref', 'Efs', 'K_ref', 'dH']   # theta1 ... theta6 of model.py

sample = {'init': 0, 'meas': True,  # every output: measured, 17 synchronised samples at least 0.3 h apart,
          'sp': 17, 'offt': 0.3,    # always at 0 and 16 h; noise sd 0.005 mol/L
          'samp_s': 1, 'samp_f': [0, 16], 'unc': 0.005}

system = {
    'tvi': {'u1': {                       # reactor temperature [K]
        'stps': 6, 'cvp': 'LPF', 'const': 'inc',   # 6 levels joined by ramps, rising only
        'min': 296.15, 'max': 306.15,
        'offl': 0.01, 'offt': 0.3,                 # min level change [K], min time between switches [h]
    }},
    'tvo': {'y1': dict(sample), 'y2': dict(sample), 'y3': dict(sample)},  # SM, product, impurity [mol/L]
    'tii': {'y10': {'min': 0.3, 'max': 1},         # initial starting material [mol/L]
            'y20': {'min': 0.19, 'max': 1}},       # initial trimethylamine [mol/L]
    'tio': {},
    't_s': [0, 16],      # 16-h batch
    't_r': 0.1 if QUICK else 0.02,   # time grid [h] (coarser in a quick check)
    't_d': 0.3,          # dead time at both ends [h] (designs only)
}

models = {
    'can_m': ['M'],
    'krt': {'M': 'pys'},                         # a Python file with solve_model(t, y0, tii, tvi, theta)
    'src': {'M': str(Path('model.py').resolve())},
    'theta': {'M': list(theta_guess)},
    't_l': {'M': theta_min},
    't_u': {'M': theta_max},
    'mutation': {'M': [True] * 6},               # True = estimated, False = held at its value
    'theta_names': {'M': NAMES},
}

In [ ]:
insilicos = {                 # four existing batches, simulated here with noise
    'tr_m': 'M', 'theta': theta_true, 'errt': 'abs',
    'prels': {
        '1': {'u1': 296.15, 'y10': 0.366, 'y20': 0.19},
        '2': {'u1': 306.15, 'y10': 0.366, 'y20': 0.19},
        '3': {'u1': 296.15, 'y10': 0.65, 'y20': 0.595},
        '4': {'u1': 306.15, 'y10': 0.65, 'y20': 0.595},
    },
}

iden_opt = {
    'meth': 'LMBFGS', 'ms': not QUICK,   # L-BFGS-B; multi-start (parallel) except in a quick check
    'ob': 'WLS', 'maxit': 60 if QUICK else 500, 'tol': 1e-8,
    'var-cov': 'M',                      # Fisher-matrix covariance (fast)
    'sens_m': 'central', 'eps': 1e-5,    # the sensitivities are also what estima ranks
    'c_plt': False, 'f_plt': True, 'plt_s': True, 'log': False,
    # 'model_update': False,             # True (default): estimate only the selected parameters and
}                                        #   fix the others; False: only report the ranking

### Add the batches one by one

Every round frees all six parameters again so that the ranking is redone with all the data.

In [ ]:
round_data, summary = {}, []
for r in (1, 2, 3, 4):
    expera(system, models, insilicos, {}, expr=r)
    models['mutation']['M'] = [True] * 6
    resultpr = parmest(system, models, iden_opt, case='strov')
    unc = uncert(resultpr, system, models, iden_opt)
    ranking, k_opt, rCC, J_k, best = estima(unc['results'], system, models, iden_opt, r)
    save_rounds(r, unc['results'], f'batches 1-{r}', round_data, models, iden_opt, unc['obs'], system,
                ranking=ranking, k_optimal_value=k_opt, rCC_values=rCC, J_k_values=J_k, best_uncert_result=best)
    summary.append({'batches': r, 'ranking (most to least estimable)': ' > '.join(NAMES[i] for i in ranking['M']),
                    'estimated': k_opt['M'], 'fixed': 6 - k_opt['M']})
pd.DataFrame(summary).set_index('batches')

### rCC against the number of parameters

The selected number is where rCC is smallest. `run_postprocessing` draws it for every round, next to
the estimates and their intervals.

In [ ]:
save_to_jac(round_data, purpose='iden')
run_postprocessing(round_data=load_from_jac()['iden'], solvers=['M'], selected_rounds=[1, 2, 3, 4],
                   plot_global_p_and_t=True, plot_confidence_spaces=False, plot_p_and_t_tests=True,
                   export_excel_reports=True, plot_estimability=True, true_theta={'M': theta_true}, show=True)

**What to take away.** One isothermal batch supports few parameters; batches at two
temperatures make the energies estimable. The parameters ranked last are the ones a designed
experiment should target (tutorial 4b), or that a simpler model should drop.

**Try:** remove batch 2 or 4 (only 296 K data) and see the energies fall to the bottom of the
ranking; set `iden_opt['model_update'] = False` to keep estimating all six.